In [1]:
import pandas as pd

In [2]:
data = pd.read_csv("../data/online_shoppers.csv")

In [3]:
data.head()

,Administrative,Administrative_Duration,Informational,Informational_Duration,ProductRelated,ProductRelated_Duration,BounceRates,ExitRates,PageValues,SpecialDay,Month,OperatingSystems,Browser,Region,TrafficType,VisitorType,Weekend,Revenue
0,0,0.0,0,0.0,1,0.000000,0.20,0.20,0.0,0.0,Feb,1,1,1,1,Returning_Visitor,False,False
1,0,0.0,0,0.0,2,64.000000,0.00,0.10,0.0,0.0,Feb,2,2,1,2,Returning_Visitor,False,False
2,0,0.0,0,0.0,1,0.000000,0.20,0.20,0.0,0.0,Feb,4,1,9,3,Returning_Visitor,False,False
3,0,0.0,0,0.0,2,2.666667,0.05,0.14,0.0,0.0,Feb,3,2,2,4,Returning_Visitor,False,False
4,0,0.0,0,0.0,10,627.500000,0.02,0.05,0.0,0.0,Feb,3,3,1,4,Returning_Visitor,True,False


In [4]:
df = data.drop_duplicates()
df.shape

(12205, 18)

In [5]:
y = df['Revenue']
X= df.drop(columns = 'Revenue')


In [6]:
from sklearn.model_selection import train_test_split

In [7]:

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.2, random_state = 42, stratify= y)

In [8]:
X_train.shape, X_test.shape, y_train.shape, y_test.shape

((9764, 17), (2441, 17), (9764,), (2441,))

In [9]:
y_train.mean()

np.float64(0.15628840639082342)

In [10]:
y_test.mean()

np.float64(0.15649324047521507)

## Split
total size after duplicated drop = (12205,18)
test rate = 0.2
Purchase rate: train 15.6%, test 15.6% (stratified)

In [11]:
from sklearn.dummy import DummyClassifier
from sklearn.metrics import accuracy_score, recall_score

In [12]:
dummy = DummyClassifier(strategy = 'most_frequent')
dummy.fit(X_train,y_train)
predictions = dummy.predict(X_train)

print(f" Accuracy : {accuracy_score(y_train, predictions):.3f} \n Recall : {recall_score(y_train,predictions)}")


 Accuracy : 0.844 
 Recall : 0.0


## Dummy baseline
- The dummy (most_frequent) always predicts majority class for every session.
- Accuracy = 84.4% because 84.4% of sessions are non-buyers, so "always no" is right that often.
- Recall = 0.0% because it never predicts minority class, so it catches none of the real buyers.
- Conclusion: accuracy is misleading on imbalanced data; we use PR-AUC instead.

In [13]:
df.groupby('Revenue')['ProductRelated_Duration'].mean()

Revenue
False    1082.976881
True     1876.209615
Name: ProductRelated_Duration, dtype: float64

In [14]:
is_zero = df['ProductRelated_Duration'] == 0     
is_zero.groupby(df['Revenue']).mean() 


Revenue
False    0.059920
True     0.006813
Name: ProductRelated_Duration, dtype: float64

## Logistic regression pipeline

In [15]:
X_train.describe()

,Administrative,Administrative_Duration,Informational,Informational_Duration,ProductRelated,ProductRelated_Duration,BounceRates,ExitRates,PageValues,SpecialDay,OperatingSystems,Browser,Region,TrafficType
count,9764.000000,9764.000000,9764.000000,9764.000000,9764.000000,9764.000000,9764.000000,9764.000000,9764.000000,9764.000000,9764.000000,9764.000000,9764.000000,9764.000000
mean,2.342687,82.219938,0.510344,34.950623,32.101803,1207.264687,0.020296,0.041469,5.945606,0.062351,2.130582,2.371876,3.134986,4.059402
std,3.349268,179.502363,1.282728,143.828323,44.817312,1926.389159,0.045234,0.046145,18.860076,0.199959,0.909649,1.734820,2.389418,4.026181
min,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1.000000,1.000000,1.000000,1.000000
25%,0.000000,0.000000,0.000000,0.000000,8.000000,193.466667,0.000000,0.014286,0.000000,0.000000,2.000000,2.000000,1.000000,2.000000
50%,1.000000,9.000000,0.000000,0.000000,18.000000,617.000000,0.002857,0.025000,0.000000,0.000000,2.000000,2.000000,3.000000,2.000000
75%,4.000000,94.725000,0.000000,0.000000,39.000000,1483.650000,0.016667,0.049242,0.000000,0.000000,3.000000,2.000000,4.000000,4.000000
max,27.000000,3398.750000,24.000000,2549.375000,705.000000,63973.522230,0.200000,0.200000,361.763742,1.000000,8.000000,13.000000,9.000000,20.000000


In [16]:
numeric_cols = ['Administrative', 'Administrative_Duration','Informational','Informational_Duration','ProductRelated','ProductRelated_Duration','BounceRates','ExitRates','PageValues','SpecialDay']
categorical_cols = ['OperatingSystems','Browser','Region','TrafficType','Weekend','Month','VisitorType']
len(numeric_cols + categorical_cols)

17

In [17]:
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression

In [21]:
preprocessor = ColumnTransformer([("num",StandardScaler(),numeric_cols),("category",OneHotEncoder(handle_unknown= 'ignore'),categorical_cols)])
